<a href="https://colab.research.google.com/github/MaiChiLe113/triageML/blob/main/data/prepare.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Triage — Step 1: `data/prepare.ipynb`
COS30018 Option D · SOC Triage ML pipeline

**Goal of this notebook:** load CICIDS2017 (MachineLearningCVE), verify schema,
clean, group 15 raw labels into 7 categories, save a reproducible cleaned dataset.

**Runs top-to-bottom on Google Colab free tier.**

Does NOT split, scale, or train. Those are later steps.

> STOP GATE: if the feature count differs from the assumption (78), the schema
> cell raises and halts. Do not proceed — report the printed columns back.

In [ ]:
# --- Setup: dependencies + reproducibility ---
# WHY:
# - kagglehub pulls the canonical CIC dataset (no manual upload, reproducible source)
# - seeds fixed so every downstream run is identical (assignment requires reproducibility)
!pip -q install kagglehub tqdm >/dev/null 2>&1

import os, random, hashlib, json
import numpy as np
import pandas as pd
from glob import glob
from tqdm.auto import tqdm

random.seed(42)
np.random.seed(42)
print("pandas", pd.__version__, "| numpy", np.__version__)

pandas 2.2.2 | numpy 2.0.2


In [ ]:
# --- Config ---
# WHY:
# - EXPECTED_N_FEATURES = 78 encodes our schema assumption; the STOP gate checks it
# - identifier cols are dropped to prevent leakage (model must not memorise ports/IDs)
# - category map is keyword-based -> robust to CICIDS punctuation variants (en-dash, spaces)
from typing import Optional

EXPECTED_N_FEATURES: int = 78
LABEL_COL: str = "Label"          # normalised (whitespace stripped) name
OUT_DIR: str = "artifacts"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs("figures", exist_ok=True)

# Columns that leak identity/time if present. MLCVE usually has none of these,
# but GeneratedLabelledFlows does. Dropped only if found.
LEAKAGE_COLS = {
    "Flow ID", "Source IP", "Src IP", "Destination IP", "Dst IP",
    "Source Port", "Src Port", "Timestamp", "Fwd Header Length.1",
}

CATEGORIES = ["benign", "dos_ddos", "port_scan", "brute_force",
              "web_attack", "botnet", "infiltration"]

In [ ]:
# --- Load data ---
# WHY:
# - MachineLearningCVE = benchmark CSV set (78 flow features + 1 label), citable in report
# - concat all 8 daily files into one frame; tqdm shows progress on Colab
# - float32 downcast keeps 2.8M rows inside free-tier RAM
import kagglehub

def load_mlcve() -> pd.DataFrame:
    """Download + concatenate the 8 MachineLearningCVE CSVs into one DataFrame."""
    root = kagglehub.dataset_download("cicdataset/cicids2017")
    csvs = sorted(glob(os.path.join(root, "**", "MachineLearningCVE", "*.csv"),
                       recursive=True))
    if not csvs:
        csvs = sorted(glob(os.path.join(root, "**", "*.csv"), recursive=True))
    assert csvs, f"No CSVs found under {root}"
    frames = []
    for f in tqdm(csvs, desc="reading CSVs"):
        frames.append(pd.read_csv(f, low_memory=False))
    df = pd.concat(frames, ignore_index=True)
    return df

df = load_mlcve()
print("raw shape:", df.shape)

KaggleApiHTTPError: 403 Client Error.

You don't have permission to access resource at URL: https://api.kaggle.com/v1/datasets.DatasetApiService/GetDataset. Please make sure you are authenticated if you are trying to access a private resource or a resource requiring consent.

In [ ]:
# --- Normalise column names ---
# WHY:
# - CICIDS2017 headers carry leading/trailing spaces (' Label', ' Flow Duration')
# - stripping once here prevents silent KeyErrors everywhere downstream
df.columns = [c.strip() for c in df.columns]
print("first 10 cols:", list(df.columns[:10]))
print("label col present:", LABEL_COL in df.columns)

In [ ]:
# --- STOP GATE: schema verification ---
# WHY:
# - assignment rule: if schema differs from assumption, STOP and report
# - counts feature cols (everything except the label); must equal EXPECTED_N_FEATURES
feature_cols = [c for c in df.columns if c != LABEL_COL]
n_feat = len(feature_cols)
print(f"feature columns detected: {n_feat} (expected {EXPECTED_N_FEATURES})")

if LABEL_COL not in df.columns:
    raise SystemExit(f"STOP: no '{LABEL_COL}' column. Columns = {list(df.columns)}")
if n_feat != EXPECTED_N_FEATURES:
    print("\nFULL COLUMN LIST:")
    for i, c in enumerate(df.columns):
        print(f"  {i:>2} {c}")
    raise SystemExit(
        f"STOP: {n_feat} feature cols != {EXPECTED_N_FEATURES}. "
        "Report this list back before continuing.")
print("schema OK")

In [ ]:
# --- Raw label distribution ---
# WHY: imbalance is the core modelling challenge; document it before any grouping
raw_counts = df[LABEL_COL].value_counts(dropna=False)
print("raw labels:", raw_counts.shape[0])
print(raw_counts)

In [ ]:
# --- Leakage scan + drop identifiers ---
# WHY:
# - flow IDs / IPs / timestamps let a model cheat by memorising, not learning traffic shape
# - 'Destination Port' is kept (legit signal) but FLAGGED for the ablation study
found_leak = [c for c in df.columns if c in LEAKAGE_COLS]
if found_leak:
    print("dropping leakage cols:", found_leak)
    df = df.drop(columns=found_leak)
else:
    print("no identity/time leakage cols found (expected for MLCVE)")

if "Destination Port" in df.columns:
    print("FLAG: 'Destination Port' retained as feature -> test with/without in ablation")

In [ ]:
# --- Handle inf / NaN, drop constant & duplicate columns ---
# WHY:
# - CICFlowMeter emits inf in rate cols (Flow Bytes/s) on zero-duration flows
# - constant cols carry no signal; duplicate cols waste compute and bias importance
def clean_numeric(frame: pd.DataFrame, label_col: str) -> pd.DataFrame:
    """Replace inf->NaN, drop rows with NaN, drop constant and duplicate feature cols."""
    feats = [c for c in frame.columns if c != label_col]
    frame[feats] = frame[feats].replace([np.inf, -np.inf], np.nan)
    before = frame.shape[0]
    frame = frame.dropna(subset=feats).reset_index(drop=True)
    print(f"dropped {before - frame.shape[0]} rows with inf/NaN")

    nunique = frame[feats].nunique()
    const_cols = nunique[nunique <= 1].index.tolist()
    if const_cols:
        print("dropping constant cols:", const_cols)
        frame = frame.drop(columns=const_cols)

    feats = [c for c in frame.columns if c != label_col]
    dup_cols = frame[feats].T.duplicated().pipe(lambda s: [feats[i] for i, v in enumerate(s) if v])
    if dup_cols:
        print("dropping duplicate cols:", dup_cols)
        frame = frame.drop(columns=dup_cols)
    return frame

df = clean_numeric(df, LABEL_COL)
# downcast after cleaning to save RAM
for c in [c for c in df.columns if c != LABEL_COL]:
    df[c] = pd.to_numeric(df[c], downcast="float")
print("clean shape:", df.shape)

In [ ]:
# --- Group 15 raw labels -> 7 categories ---
# WHY:
# - spec fixes the 7-class taxonomy; keyword matching survives '–'/'-'/spacing variants
# - ORDER matters: 'Web Attack Brute Force' must hit web_attack before brute_force
# - unknown label -> raise (never invent a mapping)
def to_category(label: str) -> str:
    """Map a raw CICIDS2017 label string to one of the 7 project categories."""
    s = str(label).strip().lower()
    if "benign" in s:                              return "benign"
    if "web attack" in s or "web-attack" in s:     return "web_attack"
    if "brute" in s or "patator" in s:             return "brute_force"
    if "portscan" in s or "port scan" in s:        return "port_scan"
    if "bot" in s:                                 return "botnet"
    if "infiltration" in s:                        return "infiltration"
    if "ddos" in s or "dos" in s or "heartbleed" in s:
        # FLAG: Heartbleed (~11 rows) has no dedicated bucket in the 7-class spec.
        # Folded into dos_ddos (same Wednesday capture, DoS-family). Override if needed.
        return "dos_ddos"
    raise ValueError(f"Unmapped label: {label!r} -> extend to_category() or STOP")

df["category"] = df[LABEL_COL].map(to_category)
cat_counts = df["category"].value_counts().reindex(CATEGORIES, fill_value=0)
print(cat_counts)
missing = [c for c in CATEGORIES if cat_counts[c] == 0]
if missing:
    print("NOTE: categories absent in this data:", missing)

In [ ]:
# --- Save category distribution figure ---
# WHY: report-ready artifact; log-scale exposes the minority classes hidden by benign
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
cat_counts.plot(kind="bar", ax=ax, color="#3b6ea5")
ax.set_yscale("log")
ax.set_ylabel("count (log)")
ax.set_title("CICIDS2017 — event count by category (7-class)")
plt.tight_layout()
fig.savefig("figures/category_distribution.png", dpi=120)
plt.close(fig)
print("saved figures/category_distribution.png")

In [ ]:
# --- Save cleaned dataset + manifest + data hash ---
# WHY:
# - parquet: compact, typed, fast reload for the split/train notebooks
# - sha256 over sorted column bytes -> data_hash for the model_card (reproducibility)
# - manifest records exact feature list so later steps never guess schema
def save_artifacts(frame: pd.DataFrame, label_col: str, out_dir: str) -> dict:
    """Write cleaned parquet + JSON manifest with feature list and data hash."""
    feats = [c for c in frame.columns if c not in (label_col, "category")]
    parquet_path = os.path.join(out_dir, "cicids2017_clean.parquet")
    frame.to_parquet(parquet_path, index=False)

    h = hashlib.sha256()
    h.update(pd.util.hash_pandas_object(frame[sorted(feats)], index=False).values.tobytes())
    data_hash = h.hexdigest()[:16]

    manifest = {
        "source": "cicids2017_MachineLearningCVE",
        "n_rows": int(frame.shape[0]),
        "n_features": len(feats),
        "feature_cols": feats,
        "label_col": label_col,
        "categories": CATEGORIES,
        "data_hash": data_hash,
    }
    with open(os.path.join(out_dir, "manifest.json"), "w") as f:
        json.dump(manifest, f, indent=2)
    return manifest

manifest = save_artifacts(df, LABEL_COL, OUT_DIR)
print(json.dumps({k: v for k, v in manifest.items() if k != "feature_cols"}, indent=2))
print("\nartifacts/ ->", os.listdir(OUT_DIR))

## Step 1 output

- `artifacts/cicids2017_clean.parquet` — cleaned, 7-class labelled
- `artifacts/manifest.json` — feature list + `data_hash`
- `figures/category_distribution.png`

**Run this in Colab. Paste back:**
1. schema cell result (78 OK, or the full column dump if it stopped)
2. the 7-class distribution
3. any error

Then Step 2 = stratified 70/15/15 split + leakage-safe scaling.